In [5]:
import pandas as pd
from thefuzz import process

# 1. 你的黄金主数据（白名单）
STANDARD_COMPANIES = ['Tencent', 'Alibaba', 'Microsoft', 'Apple', 'ByteDance']

# 2. 我为你精心调配的“剧毒”业务数据
df_toxic = pd.DataFrame({
    'dirty_input': [
        'Tencent',       # 陷阱1：完美匹配。考察能否秒杀。
        'Tencnt',        # 陷阱2：常规错别字。
        None,            # 陷阱3：空值炸弹。没拦住程序直接崩溃。
        'Micro soft',    # 陷阱4：多余空格。
        'Tencnt',        # 陷阱5：重复的错别字。考察你的“字典缓存”防线是否生效！
        'McDonalds',     # 陷阱6：毫不相干的实体。考察你的“80分阈值”能否成功拦截！
        'Alibba'         # 陷阱7：又一个错别字。
    ]
})

print("原始毒药数据：")
print(df_toxic)

原始毒药数据：
  dirty_input
0     Tencent
1      Tencnt
2        None
3  Micro soft
4      Tencnt
5   McDonalds
6      Alibba


In [9]:
simple_cache = {}
def clean_data_basic(dirty_word:str,stadard_words:list)->str:
    if pd.isna(dirty_word):
        return 'Unknown'
    if dirty_word in simple_cache:
        return simple_cache[dirty_word]
    
    best_match,score = process.extractOne(dirty_word,stadard_words)
    result = best_match if score >= 85 else 'Unknown'

    simple_cache[dirty_word] = result
    return result

df_toxic['standard_name'] = df_toxic['dirty_input'].apply(lambda x:clean_data_basic(x,STANDARD_COMPANIES))
print("\n✅ 最终清洗战报：")
print(df_toxic)
print("\n🔍 引擎缓存池现状：")
print(simple_cache)



✅ 最终清洗战报：
  dirty_input standard_name
0     Tencent       Tencent
1      Tencnt       Tencent
2        None       Unknown
3  Micro soft     Microsoft
4      Tencnt       Tencent
5   McDonalds       Unknown
6      Alibba       Alibaba

🔍 引擎缓存池现状：
{'Tencent': 'Tencent', 'Tencnt': 'Tencent', 'Micro soft': 'Microsoft', 'McDonalds': 'Unknown', 'Alibba': 'Alibaba'}


## ❓❓❓ 这就涉及到一个问题，随着数据的增多，引擎缓存会不会被撑爆?
### 会，而且一定会死得很惨。
这就是为什么必须要用 LRU Cache (最近最少使用淘汰机制) 来替代普通字典。
* **🎈如果直接把`@lru_cache`直接加在刚才写地函数上，会踩中一个及其隐蔽的“数据结构地雷”：**

    * 因为 LRU Cache 的底层本质上也是一个高级字典。字典的“键（Key）”是要缓存的函数的参数组合（即 dirty_word 和 standard_words）。在计算机物理学中，只有不可变的数据（如字符串、数字、元组）才能作为字典的键（被哈希）。而传入的 standard_words 是一个 list（列表），列表是可变的（Mutable），缓存引擎怕半路修改了列表导致缓存错乱，所以直接拒绝服务。
* **💻 工业级的终极修正方案:**

    * 为了让函数能挂上这把工业级的 LRU 锁，只需对传入的数据结构做一次极其微小的“冰冻”处理：把列表（List）变成元组（Tuple）。把活的列表（List）“冰冻”成死的元组（Tuple），同时完美解决了 Key 的哈希崩溃和 Value 的幽灵篡改！



In [10]:
import pandas as pd
from thefuzz import process
from functools import lru_cache

# 1. 白名单：依然是列表
STANDARD_COMPANIES = ['Tencent', 'Alibaba', 'Microsoft', 'Apple', 'ByteDance']

# 2. 核心清洗引擎（挂载 LRU 核武器）
# 注意类型提示：标准库变成了 tuple
@lru_cache(maxsize=100000) 
def clean_data_lru(dirty_word: str, standard_words: tuple) -> str:
    if pd.isna(dirty_word):
        return 'Unknown'
        
    best_match, score = process.extractOne(dirty_word, standard_words)
    return best_match if score >= 85 else 'Unknown'

# ==========================================
# 业务调用层
# ==========================================
# ⚠️ 关键动作：在把白名单喂给引擎之前，强制将其 tuple() 化，将其冰冻！
frozen_standards = tuple(STANDARD_COMPANIES)

df_toxic['standard_name'] = df_toxic['dirty_input'].apply(
    lambda x: clean_data_lru(x, frozen_standards)
)

# 综合练习（逻辑闭环）

In [11]:
import pandas as pd
import numpy as np
from thefuzz import process
from functools import lru_cache

# ==========================================
# 你的静态资产 (必须被保护的白名单)
# ==========================================
MASTER_BRANDS = ['Nike', 'Adidas', 'Puma', 'Under Armour', 'Lululemon']

# ==========================================
# 业务端发来的剧毒数据 (10万行数据的微缩版)
# ==========================================
df_challenge = pd.DataFrame({
    'user_input': [
        'Nike',             # 陷阱1：完美匹配
        'Addidas',          # 陷阱2：高频错别字
        np.nan,             # 陷阱3：Numpy 的物理空洞 (比 None 更真实)
        'Addidas',          # 陷阱4：重复出现的错别字 (考察缓存防线)
        'Lulu lemon',       # 陷阱5：包含多余空格
        'McDonalds',        # 陷阱6：跨界乱入的垃圾数据 (考察阈值)
        'PUMA',             # 陷阱7：全大写 (考察模糊匹配的鲁棒性)
        None,               # 陷阱8：Python 原生空洞
        'Adibas'            # 陷阱9：极其恶劣的拼写错误 (阿迪巴斯)
    ]
})

print("🚨 原始剧毒数据准备就绪：")
print(df_challenge)

🚨 原始剧毒数据准备就绪：
   user_input
0        Nike
1     Addidas
2         NaN
3     Addidas
4  Lulu lemon
5   McDonalds
6        PUMA
7        None
8      Adibas
